# Week 11: Advanced Specialized Machine Learning
## Notebook 11: Matrix Factorization: Alternating Least Squares (ALS)

### 1. Conceptual Intuition & Theoretical Background
In real-world recommender systems (e.g. Netflix, Spotify), user interactions are **implicit** (clicks, watch time, listens) rather than explicit ratings (1 to 5 stars). A zero does not mean the user dislikes the item; it may mean they simply haven't discovered it yet.

**Implicit Feedback ALS (Hu, Koren, Volinsky, 2008):**
1. **Binary Preference ($p_{ui}$):**
$$p_{ui} = \begin{cases} 1 & \text{if } r_{ui} > 0 \\ 0 & \text{if } r_{ui} = 0 \end{cases}$$
2. **Confidence Weighting ($c_{ui}$):**
$$c_{ui} = 1 + \alpha r_{ui}$$
Where $\alpha$ is a rate hyperparameter (e.g. $\alpha = 40$).

**Objective Function:**
$$\mathcal{L} = \sum_{u, i} c_{ui} (p_{ui} - x_u^T y_i)^2 + \lambda \left( \sum_u \|x_u\|^2 + \sum_i \|y_i\|^2 \right)$$

**Alternating Updates (Closed-Form):**
Fixing item factors $Y$ and solving for user factor $x_u$:
$$x_u = \left( Y^T C^u Y + \lambda I \right)^{-1} Y^T C^u p_u$$
Where $C^u = \text{diag}(c_{u1}, \dots, c_{um})$.

---

### 2. Scratch Python Implementation: Implicit ALS Engine

In [ ]:
import numpy as np

class ImplicitALSScratch:
    def __init__(self, n_factors=8, reg_lambda=0.1, alpha=40, n_iter=10):
        self.n_factors = n_factors
        self.reg_lambda = reg_lambda
        self.alpha = alpha
        self.n_iter = n_iter
        self.user_factors = None
        self.item_factors = None
        
    def fit(self, R):
        """R: User-Item interaction matrix (N_users, N_items)"""
        n_users, n_items = R.shape
        f = self.n_factors
        
        # Preference matrix P and Confidence matrix C
        P = (R > 0).astype(np.float64)
        C = 1.0 + self.alpha * R
        
        # Initialize latent factors randomly
        np.random.seed(42)
        self.user_factors = np.random.normal(0, 0.05, (n_users, f))
        self.item_factors = np.random.normal(0, 0.05, (n_items, f))
        
        lambda_I = self.reg_lambda * np.eye(f)
        
        for iteration in range(self.n_iter):
            # 1. Update User Factors: x_u = (Y^T C^u Y + lambda I)^(-1) Y^T C^u p_u
            YtY = np.dot(self.item_factors.T, self.item_factors)
            for u in range(n_users):
                Cu = C[u, :]
                # Fast diagonal update: Y^T (C^u - I) Y + Y^T Y
                Cu_minus_I = Cu - 1.0
                Yt_Cu_Y = YtY + np.dot(self.item_factors.T * Cu_minus_I, self.item_factors)
                
                # Y^T C^u p_u
                Yt_Cu_pu = np.dot(self.item_factors.T * Cu, P[u, :])
                self.user_factors[u] = np.linalg.solve(Yt_Cu_Y + lambda_I, Yt_Cu_pu)
                
            # 2. Update Item Factors: y_i = (X^T C^i X + lambda I)^(-1) X^T C^i p_i
            XtX = np.dot(self.user_factors.T, self.user_factors)
            for i in range(n_items):
                Ci = C[:, i]
                Ci_minus_I = Ci - 1.0
                Xt_Ci_X = XtX + np.dot(self.user_factors.T * Ci_minus_I, self.user_factors)
                
                Xt_Ci_pi = np.dot(self.user_factors.T * Ci, P[:, i])
                self.item_factors[i] = np.linalg.solve(Xt_Ci_X + lambda_I, Xt_Ci_pi)
                
        return self
        
    def predict(self):
        return np.dot(self.user_factors, self.item_factors.T)

# Benchmark on 5 Users x 6 Items Synthetic Interaction Stream
interaction_matrix = np.array([
    [10, 0, 5, 0, 0, 1],
    [0, 15, 0, 8, 2, 0],
    [1, 0, 12, 0, 0, 4],
    [0, 7, 0, 14, 0, 0],
    [8, 0, 6, 0, 0, 2]
], dtype=float)

als = ImplicitALSScratch(n_factors=3, reg_lambda=0.01, alpha=20, n_iter=8)
als.fit(interaction_matrix)
predicted_scores = als.predict()

print("--- Implicit Feedback ALS Recommender ---")
print(f"User 0 Observed Interactions: {interaction_matrix[0]}")
print(f"User 0 Predicted Scores:       {np.round(predicted_scores[0], 2)}")
print(f"Top Recommended Item for User 0: Item {np.argmax(predicted_scores[0] * (interaction_matrix[0] == 0))}")

### 3. Step-by-Step Code Walkthrough
1. **Confidence Scaling ($c_{ui} = 1 + \alpha r_{ui}$):** Higher engagement (e.g. repeated video plays) translates into higher fitting confidence.
2. **$Y^T Y$ Precomputation:** Precomputes $Y^T Y$ globally to reduce per-user matrix construction costs.
3. **Closed-Form Quadratic Step:** Solves positive-definite linear systems using `np.linalg.solve` rather than iterative SGD.

---

### 4. Industry Applications & Trade-offs
- **Spotify & E-Commerce Collaborative Filtering:** Matrix factorization on purchase history and streaming logs.
- **Trade-off:** Suffers from the **Cold-Start Problem** for new users or items with zero historical interaction data.

---

### 5. Top Interview Questions & Answers
**Q1: Why is Alternating Least Squares (ALS) preferred over Stochastic Gradient Descent (SGD) for implicit feedback datasets?**
> **Answer:** In implicit feedback, all unobserved entries are treated as negatives with low confidence, making the target matrix fully dense ($N \times M$). ALS computes global closed-form updates efficiently via $Y^T Y$ caching, whereas SGD must sample negative instances.

**Q2: What is the purpose of the $\alpha$ confidence multiplier in implicit ALS?**
> **Answer:** It governs how aggressively the model trusts positive engagement signals over implicit negative assumptions. Higher $\alpha$ assigns greater weight to frequently consumed items.